# Quantum-Assisted Graph Optimization for LEO Constellation Design

**Authors:** Based on Owens-Fahrner, N., Wysack, J., Kim, J. (2025)  
**Date:** 2025

---

## Problem Context

Low Earth Orbit (LEO) is increasingly congested. As of July 2025, LEO contains over **22,000 tracked objects**, including nearly **9,000 debris fragments** from decades of launches and fragmentation events. The rapid deployment of mega-constellations — Starlink (~6,000 satellites), OneWeb, Amazon Kuiper — has dramatically accelerated this trend.

Designing a satellite constellation in this environment requires carefully balancing two competing objectives:

1. **Collision risk minimization** — avoid orbital shells crowded with debris
2. **Coverage maximization** — ensure the constellation can observe and track LEO objects effectively

This notebook demonstrates the methodology of **Owens-Fahrner et al. (2025)**: formulating constellation selection as a **Densest k-Subgraph (DkS) problem** on a weighted graph, then solving it using both classical simulated annealing and D-Wave quantum annealing via a **QUBO (Quadratic Unconstrained Binary Optimization)** formulation.

**What this notebook shows:**
- How to construct the satellite candidate graph
- How to derive the QUBO Q matrix from the graph
- How to solve the QUBO with simulated annealing (CPU, no credentials needed)
- How to solve the QUBO with D-Wave quantum annealing (QPU, requires Leap account)
- How to compare and interpret the results

In [ ]:
import os
import sys

# Add project root to path so src/ modules are importable
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import networkx as nx

from config.settings import DEFAULT_K, SA_NUM_READS
from src.graph_builder import build_graph, graph_summary, compute_edge_weight
from src.qubo_formulator import build_qubo, qubo_to_bqm, evaluate_solution, print_qubo_stats
from src.classical_annealing import solve_simulated_annealing, print_results, filter_feasible
from src.quantum_annealing import check_leap_access, solve_quantum, compare_solvers

print('Setup complete.')

## 1. Satellite Data

In [ ]:
satellites_df = pd.read_csv('../data/sample_satellites.csv')
print(f'Loaded {len(satellites_df)} candidate satellites across {satellites_df["shell"].nunique()} orbital shells.\n')
display(satellites_df.head(10))
display(satellites_df.describe())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

shell_colors = {1: '#3498db', 2: '#2ecc71', 3: '#e74c3c', 4: '#f39c12', 5: '#9b59b6'}
shell_labels = {
    1: 'Shell 1 (~400 km, ~15°)',
    2: 'Shell 2 (~475 km, ~60°)',
    3: 'Shell 3 (~550 km, ~30°) — Starlink density',
    4: 'Shell 4 (~625 km, ~45°)',
    5: 'Shell 5 (~700 km, ~75°)',
}

for shell in sorted(satellites_df['shell'].unique()):
    subset = satellites_df[satellites_df['shell'] == shell]
    ax.scatter(
        subset['altitude_km'], subset['pc'],
        c=shell_colors[shell],
        s=subset['coverage'] * 300,
        alpha=0.8,
        label=shell_labels[shell],
        edgecolors='black', linewidth=0.5
    )
    for _, row in subset.iterrows():
        ax.annotate(row['satellite_id'], (row['altitude_km'], row['pc']),
                    fontsize=7, ha='left', va='bottom')

ax.set_xlabel('Altitude (km)', fontsize=12)
ax.set_ylabel('Collision Probability Pc', fontsize=12)
ax.set_title('Candidate Satellites: Risk vs Altitude\n(bubble size ∝ coverage fraction)', fontsize=13)
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Graph Construction

We build a **complete weighted graph** K_N where:
- Each **node** is a candidate satellite
- Each **edge weight** encodes the joint desirability of including both satellites:

$$w(v_n, v_m) = x \cdot (1 - P_{c,n})(1 - P_{c,m}) + y \cdot \frac{a_n + a_m}{2}$$

The Densest k-Subgraph problem then selects k nodes whose **induced subgraph** has maximum total internal edge weight.

In [ ]:
k = DEFAULT_K
G = build_graph(satellites_df)
graph_summary(G, k)

weights = [d['weight'] for _, _, d in G.edges(data=True)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(weights, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('Edge weight w(vn, vm)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Distribution of Edge Weights in the Satellite Graph', fontsize=12)
ax.axvline(np.mean(weights), color='red', linestyle='--', label=f'Mean = {np.mean(weights):.3f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
pos = nx.spring_layout(G, seed=42, k=1.5)

weights = [d['weight'] for _, _, d in G.edges(data=True)]
min_w, max_w = min(weights), max(weights)
edge_colors = [
    cm.YlOrRd(0.2 + 0.8 * (d['weight'] - min_w) / (max_w - min_w + 1e-9))
    for _, _, d in G.edges(data=True)
]

# Colour nodes by shell
node_colors = [shell_colors[G.nodes[n]['shell']] for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=400, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=1.0, alpha=0.6, ax=ax)

sm = plt.cm.ScalarMappable(cmap=cm.YlOrRd, norm=plt.Normalize(vmin=min_w, vmax=max_w))
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Edge weight')
ax.set_title('Complete Satellite Graph K_20\n(node colour = orbital shell, edge colour = weight)', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()

## 3. QUBO Formulation

We convert the Densest k-Subgraph problem to a QUBO (Quadratic Unconstrained Binary Optimization) that D-Wave samplers can solve directly.

**Objective with cardinality constraint:**
$$\min \; -\sum_{n<m} x_n x_m w(v_n, v_m) + P\left(\sum_n x_n - k\right)^2$$

**Q matrix entries** (using $x_n^2 = x_n$ for binary variables):

| Entry | Formula |
|---|---|
| Diagonal $Q_{nn}$ | $P(1 - 2k)$ |
| Off-diagonal $Q_{nm}$ | $-w(v_n,v_m) + 2P$ |

**Penalty constant:** $P = \text{multiplier} \times w_{\max} \times k^2$

The penalty $P$ must be large enough that any infeasible solution (sum ≠ k) has higher energy than any feasible one. If P is too small, the solver may prefer violating the constraint.

In [ ]:
Q, node_idx, P = build_qubo(G, k)
print_qubo_stats(Q, node_idx, k, P)

# Build dense matrix for visualisation (first 10 nodes only)
n_show = 10
nodes_show = list(G.nodes())[:n_show]
idx_show = {s: node_idx[s] for s in nodes_show}

Q_dense = np.zeros((n_show, n_show))
for (i, j), val in Q.items():
    if i < n_show and j < n_show:
        Q_dense[i, j] = val
        if i != j:
            Q_dense[j, i] = val

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    Q_dense, annot=True, fmt='.2f',
    xticklabels=nodes_show, yticklabels=nodes_show,
    cmap='RdBu_r', center=0, ax=ax
)
ax.set_title(f'QUBO Q Matrix (first {n_show} nodes, k={k})', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Verify penalty: infeasible solution should have higher energy than feasible one
import random

all_indices = list(range(len(node_idx)))

# Feasible: exactly k = DEFAULT_K satellites selected
selected_k = random.sample(all_indices, k)
feasible_sample = {i: (1 if i in selected_k else 0) for i in all_indices}

# Infeasible: select k+2 satellites
selected_k2 = random.sample(all_indices, k + 2)
infeasible_sample = {i: (1 if i in selected_k2 else 0) for i in all_indices}

from src.qubo_formulator import evaluate_solution
res_f = evaluate_solution(feasible_sample, Q, node_idx, k)
res_inf = evaluate_solution(infeasible_sample, Q, node_idx, k)

print(f'Feasible solution   (k={res_f["num_selected"]}): energy = {res_f["energy"]:.4f},   feasible = {res_f["feasible"]}')
print(f'Infeasible solution (k={res_inf["num_selected"]}): energy = {res_inf["energy"]:.4f}, feasible = {res_inf["feasible"]}')
print()
if res_inf['energy'] > res_f['energy']:
    print('✓ Penalty is correctly calibrated: infeasible solution has HIGHER energy.')
else:
    print('✗ Warning: infeasible energy <= feasible energy. Consider increasing PENALTY_MULTIPLIER.')

## 4. Simulated Annealing

In [ ]:
bqm = qubo_to_bqm(Q)

print('Running Simulated Annealing (100 reads)...')
results_sa = solve_simulated_annealing(bqm, node_idx, k, num_reads=100, num_sweeps=1000)
print_results(results_sa, satellites_df)

In [ ]:
# Energy distribution across all reads
sampleset = results_sa['sampleset']
all_energies = [e for _, e in sampleset.data(['sample', 'energy'])]
feasible_pairs = filter_feasible(sampleset, node_idx, k)
feasible_energies = [e for _, e in feasible_pairs]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(all_energies, bins=30, color='lightsteelblue', edgecolor='white', alpha=0.9, label='All reads')
ax.hist(feasible_energies, bins=20, color='#2ecc71', edgecolor='white', alpha=0.8, label=f'Feasible (k={k})')
ax.axvline(results_sa['best_energy'], color='red', linestyle='--',
           label=f'Best feasible: {results_sa["best_energy"]:.4f}')
ax.set_xlabel('QUBO Energy', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title(f'SA Energy Distribution (100 reads, k={k})', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print(f'Feasibility rate: {results_sa["feasibility_rate"]:.1%} '
      f'({results_sa["num_feasible"]} feasible / 100 reads)')

## 5. Quantum Annealing (D-Wave Leap)

### The Adiabatic Process

D-Wave quantum annealing implements an **adiabatic evolution** between two Hamiltonians:

$$H(s) = A(s) \cdot H_A + B(s) \cdot H_B$$

where $s \in [0, 1]$ is the normalized anneal fraction:

| Phase | $A(s)$ | $B(s)$ | Physical meaning |
|---|---|---|---|
| $s=0$ | Large | ~0 | Quantum tunneling dominates; all qubits in superposition |
| $0<s<1$ | Decreasing | Increasing | Gradual handoff from quantum to classical |
| $s=1$ | ~0 | Large | Problem Hamiltonian dominates; qubits collapse to {0,1} |

**Key advantage**: quantum tunneling allows the system to traverse energy barriers that would trap classical simulated annealing in local minima. For problems with a large **minimum spectral gap** (energy gap between ground and first excited state), quantum annealing is theoretically guaranteed to find the global optimum.

**Minor-embedding**: the logical QUBO graph K_20 is denser than the physical QPU connectivity (Pegasus graph). `EmbeddingComposite` maps each logical variable to a **chain** of physical qubits, with ferromagnetic couplings enforcing chain consistency.

In [ ]:
results_qa = None
leap_available = check_leap_access()

if leap_available:
    print('D-Wave Leap is configured. Running quantum annealing...')
    try:
        results_qa = solve_quantum(bqm, node_idx, k, num_reads=100)
        print_results(results_qa, satellites_df)
        if 'timing' in results_qa and results_qa['timing']:
            print('\nQPU timing:', results_qa['timing'])
        if results_qa.get('solver_info'):
            print('QPU solver info:', results_qa['solver_info'])
    except Exception as e:
        print(f'Quantum annealing failed: {e}')
else:
    print()
    print('Leap not configured. To enable quantum annealing, run:')
    print('  dwave config create')
    print()
    print('Continuing with classical results only.')

In [ ]:
if results_qa is not None:
    compare_solvers(results_sa, results_qa, satellites_df)
else:
    print('Quantum annealing not available — skipping comparison.')

## Conclusions

This notebook demonstrated the full pipeline from **Owens-Fahrner et al. (2025)**:

1. **Graph construction**: encoding satellite safety and coverage as edge weights in K_N.
2. **QUBO formulation**: converting Densest k-Subgraph to an energy minimization problem with a penalty-enforced cardinality constraint.
3. **Classical solving**: simulated annealing and tabu search provide good solutions without any cloud credentials.
4. **Quantum solving**: D-Wave QPU leverages quantum tunneling to potentially find better solutions, especially as problem size grows.

### Limitations

- **Synthetic data**: the sample dataset uses pre-computed Pc and coverage values. A production system would require full orbital propagation (SGP4 + TLE catalog from Space-Track.org) and conjunction geometry computation at each time step.
- **Coverage model**: the five access constraints (LOS, solar exclusion, Earth limb, target sunlit, SNR) are approximated here. Full implementation requires integration with STK, GMAT, or Orekit.
- **Problem scale**: current D-Wave QPUs support O(5000) qubits. For N=20 satellites, K_20 requires 190 logical couplings — well within hardware limits. For N=100+, embedding becomes the bottleneck.

### Future Work

- Integration with the **ORDEM debris model** (NASA) for higher-fidelity collision probability estimates
- Testing on the **real TLE catalog** from Space-Track.org with actual conjunction data messages (CDMs)
- Longer simulation periods and full ephemeris propagation
- Comparison with QAOA (Quantum Approximate Optimization Algorithm) on gate-model QPUs
- Benchmarking QPU vs SA performance as a function of N (problem size scaling)